# 02. Дообучение e5-base

Дообучаю e5-base на парах запрос -> выбранное объявление из train, чтобы энкодер лучше понимал запросы к услугам. К каждой паре добавляю два трудных негатива: похожие по BM25 объявления, которые по этому запросу не выбирали. Потом считаю эмбеддинги всех объявлений и запросов для пайплайна.

- Входы: те же, что у ноутбука 01, и `artifacts/ft_train.parquet` (его готовит `python -m src.hardneg`).
- Выходы: модель и эмбеддинги в `MyDrive/avito_dataset/out/ft/e5-base/` и архив `e5-base-ft.zip`, его нужно распаковать в `artifacts/dense/`.
- Время на T4: около двух часов, из них обучение около 95 минут.

Сокращения те же, что в ноутбуке 01: `tpd` - текст объявления из заголовка, параметров и начала описания; `q` и `qf` - запрос без фильтров и с фильтрами; `lf` - поиск сначала в локальной зоне, `glob` и `loc` - по всему корпусу и только в зоне.

In [ ]:
# версии, с которыми считались эмбеддинги
!pip -q install sentence-transformers==5.2.2 transformers==5.1.0 datasets==5.0.1 accelerate==1.15.0
import os, sys, torch
print(sys.version, "torch", torch.__version__)
assert torch.cuda.is_available(), "Нужен GPU: Runtime -> Change runtime type -> T4 GPU"
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
DRIVE = "/content/drive/MyDrive/avito_dataset"
OUT = f"{DRIVE}/out/ft"
os.makedirs(OUT, exist_ok=True)
print(os.listdir(DRIVE))

## Тексты запросов и объявлений
Та же логика, что `clean`, `item_text` и `query_text` в `src/text.py`.

In [ ]:
# Тексты для e5, та же логика, что clean, item_text и query_text в src/text.py.
import re

DESC_WORDS = 120

_LAT2CYR = str.maketrans("aeopcxykmthb", "аеорсхукмтнв")
_CYR2LAT = str.maketrans("аеорсхукмтнв", "aeopcxykmthb")
_CYR = re.compile(r"[а-я]")
_LAT = re.compile(r"[a-z]")
_LAT_ONLY = re.compile(r"[dfgijlnqrsuvwz]")
_CYR_ONLY = re.compile(r"[бгдежзийлпфцчшщъыьэюя]")
_WORD = re.compile(r"\w+")
_EMOJI = re.compile("[\U0001F000-\U0001FAFF☀-➿⬀-⯿️‍⃣]+")
_REPEAT_PUNCT = re.compile(r"([^\w\s])\1+")
_SPACES = re.compile(r"\s+")


def _fix_word(m):
    w = m.group(0)
    if not (_CYR.search(w) and _LAT.search(w)):
        return w
    lat_only, cyr_only = bool(_LAT_ONLY.search(w)), bool(_CYR_ONLY.search(w))
    if cyr_only and not lat_only:
        return w.translate(_LAT2CYR)
    if lat_only and not cyr_only:
        return w.translate(_CYR2LAT)
    return w.translate(_LAT2CYR) if len(_CYR.findall(w)) >= len(_LAT.findall(w)) else w.translate(_CYR2LAT)


def clean(s: str) -> str:
    s = (s or "").lower().replace("ё", "е")
    if _CYR.search(s) and _LAT.search(s):
        s = _WORD.sub(_fix_word, s)
    s = _EMOJI.sub(" ", s)
    s = _REPEAT_PUNCT.sub(r"\1", s)
    return _SPACES.sub(" ", s).strip()


_VID = re.compile(r"Вид услуги (.+?)(?= Место оказания услуг| Тип услуги| Тип стоимости| Вид услуги|$)")
_TIP = re.compile(r"Тип услуги (?:автосервиса )?(.+?)(?= Место оказания услуг| Вид услуги| Тип услуги| Тип стоимости|$)")
_NAMES = re.compile(r"(?:Название услуги|Услуги) (.+?)(?= Начальная цена| Тип стоимости| Стоимость| Продолжительность|$)")


# Значение обрезаем на первом слове с заглавной буквы: с него начинается следующий ключ.
def _cut(v: str, max_words: int = 8) -> str:
    words = v.split()
    out = words[:1]
    for w in words[1:max_words]:
        if w[:1].isupper():
            break
        out.append(w)
    return " ".join(out)


# Из параметров только вид, тип и до восьми названий услуг.
def params_short(p: str, max_names: int = 8) -> str:
    p = p or ""
    vid = [_cut(v) for v in _VID.findall(p) if v and not v.startswith(("Место", "Тип"))]
    tip = [_cut(v) for v in _TIP.findall(p) if v and not v.startswith(("Место", "Вид"))]
    names = []
    for n in _NAMES.findall(p):
        n = _cut(n.strip())
        if n and n not in names and len(n) < 80:
            names.append(n)
        if len(names) >= max_names:
            break
    parts = []
    if vid:
        parts.append("вид услуги: " + ", ".join(dict.fromkeys(vid)))
    if tip:
        parts.append("тип услуги: " + ", ".join(dict.fromkeys(tip)))
    if names:
        parts.append("услуги: " + "; ".join(names))
    return clean(". ".join(parts))


# rep: t - заголовок, tp - ещё параметры, tpd - ещё начало описания (отсюда items_tpd.npy).
def item_text(title: str, params: str, desc: str, rep: str = "tpd") -> str:
    parts = [clean(title)]
    if rep in ("tp", "tpd"):
        ps = params_short(params)
        if ps:
            parts.append(ps)
    if rep == "tpd":
        d = " ".join(clean(desc).split()[:DESC_WORDS])
        if d:
            parts.append(d)
    return ". ".join(parts)


# с with_filters к запросу дописываются фильтры: q_<часть>_qf.npy, без них q_<часть>_q.npy
def query_text(query: str, filters: str = "", with_filters: bool = False) -> str:
    q = clean(query)
    if with_filters and filters:
        q = q + ". " + clean(filters)
    return q

In [ ]:
import numpy as np, pandas as pd, pyarrow.parquet as pq, time, gc
SEED = 42
MODEL_KEY = "e5-base"
HF = "intfloat/multilingual-e5-base"
QP, PP = "query: ", "passage: "
# запрос с фильтрами: так e5 давал больший прирост к BM25
QUERY_VARIANT = "qf"
REP = "tpd"
LR, BS, EPOCHS, WARMUP = 2e-5, 48, 1, 0.1
ITEM_MAX_LEN, QUERY_MAX_LEN = 192, 64
KS = [50, 300, 1000]
ALL_RUSSIA = 621540
MOUT = f"{OUT}/{MODEL_KEY}"
os.makedirs(MOUT, exist_ok=True)
# Чекпоинты на Диске, чтобы пережить обрыв сессии. Нужно около 7 ГБ: старый чекпоинт уходит в корзину.
CKPT_DIR = f"{MOUT}/ckpt"

## Обучающие данные

In [ ]:
# тексты строит тот же item_text, что и при оценке
ft = pd.read_parquet(f"{DRIVE}/ft_train.parquet")
neg_cols = [c for c in ft.columns if c.startswith("neg")]
need_ids = sorted(set(ft.pos) | set(ft[neg_cols].values.ravel()))
ICOLS = ["item_id", "item_title_raw", "item_infm_params_text", "item_description_raw"]
tr = pq.read_table(f"{DRIVE}/train.parquet", columns=ICOLS, filters=[("item_id", "in", need_ids)]).to_pandas().drop_duplicates("item_id")
txt = dict(zip(tr.item_id, (item_text(t, p, d, REP) for t, p, d in zip(tr.item_title_raw, tr.item_infm_params_text, tr.item_description_raw))))
del tr; gc.collect()
from datasets import Dataset
train_ds = Dataset.from_dict({
    "anchor": [QP + query_text(q, f, QUERY_VARIANT == "qf") for q, f in zip(ft.search_query, ft.search_infm_params_text)],
    "positive": [PP + txt[i] for i in ft.pos],
    **{f"negative_{k+1}": [PP + txt[i] for i in ft[c]] for k, c in enumerate(neg_cols)},
}).shuffle(seed=SEED)
print(train_ds)
print(train_ds[0])

## Обучение

In [ ]:
from sentence_transformers import SentenceTransformer, SentenceTransformerTrainer, SentenceTransformerTrainingArguments, losses
from sentence_transformers.training_args import BatchSamplers
from transformers.trainer_utils import get_last_checkpoint
import shutil

final_dir = f"{MOUT}/final"
if os.path.exists(f"{final_dir}/config.json"):
    print("готовая модель уже есть, обучение пропускаем:", final_dir)
else:
    model = SentenceTransformer(HF, device="cuda")
    model.max_seq_length = ITEM_MAX_LEN
    loss = losses.CachedMultipleNegativesRankingLoss(model, mini_batch_size=16)
    args = SentenceTransformerTrainingArguments(
        output_dir=CKPT_DIR, num_train_epochs=EPOCHS, per_device_train_batch_size=BS,
        learning_rate=LR, warmup_ratio=WARMUP, fp16=True,
        batch_sampler=BatchSamplers.NO_DUPLICATES,
        # чекпоинт тяжёлый: пишем редко и храним только последний
        save_strategy="steps", save_steps=1000, save_total_limit=1, logging_steps=50,
        seed=SEED, report_to="none", dataloader_drop_last=True,
    )
    trainer = SentenceTransformerTrainer(model=model, args=args, train_dataset=train_ds, loss=loss)
    last = get_last_checkpoint(CKPT_DIR) if os.path.isdir(CKPT_DIR) else None
    t0 = time.time()
    trainer.train(resume_from_checkpoint=last)
    print(f"обучение: {time.time()-t0:.0f}s")
    model.save(final_dir)
    pd.DataFrame(trainer.state.log_history).to_csv(f"{MOUT}/train_log.csv", index=False)
    del trainer, model; gc.collect()
    # модель уже в final/, чекпоинты удаляем (корзину Диска надо очистить вручную)
    shutil.rmtree(CKPT_DIR, ignore_errors=True)
    torch.cuda.empty_cache()

## Оценка на val и эмбеддинги для пайплайна

In [ ]:
# Объявления: корпус бенчмарка и ответы val и rank, которых в нём нет.
# Их порядок - порядок строк в items_*.npy, он сохраняется в items_order.parquet.
t0 = time.time()
corp = pd.read_parquet(f"{DRIVE}/corpora.parquet")
need = corp[corp.in_bench | corp.in_val | corp.in_rank]
ICOLS = ["item_id", "item_title_raw", "item_infm_params_text", "item_description_raw", "item_location_id", "item_category_id"]
bi = pd.read_parquet(f"{DRIVE}/benchmark_items.parquet", columns=ICOLS)
extra_ids = sorted(set(need.item_id) - set(bi.item_id))
tr_items = pq.read_table(f"{DRIVE}/train.parquet", columns=ICOLS, filters=[("item_id", "in", extra_ids)]).to_pandas().drop_duplicates("item_id")
items = pd.concat([bi, tr_items], ignore_index=True).drop_duplicates("item_id").reset_index(drop=True)
items = items.merge(need, on="item_id", how="left")
for c in ["in_bench", "in_val", "in_rank"]:
    items[c] = items[c].fillna(False).astype(bool)
del bi, tr_items; gc.collect()

sq = pd.read_parquet(f"{DRIVE}/split_queries.parquet")
queries = {"val": sq[sq.part == "val"].reset_index(drop=True), "rank": sq[sq.part == "rank"].reset_index(drop=True),
           "bench": pd.read_parquet(f"{DRIVE}/benchmark_queries.parquet")}
ITEM_IDX = {iid: i for i, iid in enumerate(items.item_id)}
print(f"items {len(items):,}, queries " + ", ".join(f"{k} {len(v)}" for k, v in queries.items()), f"{time.time()-t0:.0f}s")

In [ ]:
# Локальные зоны запросов, как в src/split.py: регион - локации, дающие 90% выборов по base.
pairs = pd.read_parquet(f"{DRIVE}/train.parquet", columns=["search_location_id", "item_id", "item_location_id"])
split = pd.read_parquet(f"{DRIVE}/split.parquet", columns=["row_id", "part"]).sort_values("row_id")
base = pairs[split.part.values == "base"]
item_locs = set(pairs.item_location_id.unique()) | set(items.item_location_id.unique())
region_locs = set(base.search_location_id.unique()) - item_locs
region_map = {}
for L in region_locs:
    if L == ALL_RUSSIA:
        region_map[L] = None
        continue
    s = base.loc[base.search_location_id == L, "item_location_id"].value_counts(normalize=True)
    region_map[L] = np.array(s.index[: int(np.searchsorted(s.cumsum().values, 0.9)) + 1])
del pairs, split, base; gc.collect()
print("регионов:", len(region_map))

In [ ]:
from sentence_transformers import SentenceTransformer

def item_texts(rep):
    return [item_text(t, p, d, rep) for t, p, d in zip(items.item_title_raw, items.item_infm_params_text, items.item_description_raw)]

def query_texts(df, variant):
    return [query_text(q, f, variant == "qf") for q, f in zip(df.search_query, df.search_infm_params_text)]

def load_model(hf):
    return SentenceTransformer(hf, device="cuda").half()

# готовый .npy на Диске не пересчитываем, так переживаем обрыв сессии
def encode_cached(path, model, texts, prefix, max_len, bs):
    if os.path.exists(path):
        return np.load(path)
    model.max_seq_length = max_len
    t0 = time.time()
    emb = model.encode([prefix + t for t in texts], batch_size=bs, normalize_embeddings=True,
                       convert_to_numpy=True, show_progress_bar=True).astype(np.float16)
    np.save(path, emb)
    print(f"  {os.path.basename(path)}: {emb.shape}, {time.time()-t0:.0f}s")
    return emb

In [ ]:
# Маска локальной зоны (запросы x объявления): город - та же локация, регион - его локации, вся Россия - всё.
def local_mask(qloc, loc_t):
    rows = []
    for L in qloc:
        if L in region_map:
            area = region_map[L]
            rows.append(torch.ones_like(loc_t, dtype=torch.bool) if area is None else torch.isin(loc_t, torch.as_tensor(area, device=loc_t.device)))
        else:
            rows.append(loc_t == int(L))
    return torch.stack(rows)

# Топ-K в режимах glob, loc и lf; индексы в порядке items, -1 - пусто.
def search(q_emb, qdf, item_emb, corpus_mask, K=1000, bs=256):
    cidx = np.flatnonzero(corpus_mask)
    I = torch.from_numpy(item_emb[cidx]).to("cuda").half()
    loc_t = torch.from_numpy(items.item_location_id.values[cidx].astype(np.int64)).to("cuda")
    not114 = torch.from_numpy(items.item_category_id.values[cidx] != 114).to("cuda")
    K = min(K, len(cidx))
    out = {m: np.full((len(qdf), K), -1, dtype=np.int64) for m in ["glob", "loc", "lf"]}
    for s in range(0, len(qdf), bs):
        sub = qdf.iloc[s:s + bs]
        Q = torch.from_numpy(q_emb[s:s + bs]).to("cuda").to(I.dtype)
        S = (Q @ I.T).float()
        # фильтр категории: запросам категории 114 только её объявления
        cat114 = torch.from_numpy(sub.search_category.values == 114).to("cuda")
        S[cat114[:, None] & not114[None, :]] = -1e4
        L = local_mask(sub.search_location_id.values, loc_t)
        for mode, M in [("glob", S), ("lf", S + 10.0 * L), ("loc", torch.where(L, S, torch.full_like(S, -1e4)))]:
            v, ix = M.topk(K, dim=1)
            ix = cidx[ix.cpu().numpy()]
            ix[(v < -1e3).cpu().numpy()] = -1
            out[mode][s:s + len(sub)] = ix
    return out

def recall(lists, gold, k):
    return float(np.mean([len(set(l[:k][l[:k] >= 0]) & g) / len(g) for l, g in zip(lists, gold)]))

val = queries["val"]
gold = [set(ITEM_IDX[i] for i in a if i in ITEM_IDX) for a in val.answers]
# recall во всех трёх режимах
def metric_row(res, **info):
    row = dict(info)
    for mode, lists in res.items():
        for k in KS:
            row[f"{mode}@{k}"] = round(recall(lists, gold, k), 4)
    return row

In [ ]:
metrics_path = f"{MOUT}/metrics.csv"
model = load_model(final_dir)
qemb = {}
for part, df in queries.items():
    for variant in ["q", "qf"]:
        qemb[(part, variant)] = encode_cached(f"{MOUT}/q_{part}_{variant}.npy", model, query_texts(df, variant), QP, QUERY_MAX_LEN, 512)
items[["item_id"]].to_parquet(f"{MOUT}/items_order.parquet", index=False)
t0 = time.time()
iemb = encode_cached(f"{MOUT}/items_{REP}.npy", model, item_texts(REP), PP, ITEM_MAX_LEN, 128)
t_enc = time.time() - t0
rows = []
for variant in ["q", "qf"]:
    res = search(qemb[("val", variant)], val, iemb, items.in_val.values)
    rows.append(metric_row(res, model=f"{MODEL_KEY}-ft", rep=REP, query=variant, t_encode_items_s=round(t_enc)))
    print(rows[-1])
df = pd.DataFrame(rows)
df.to_csv(metrics_path, index=False)
pd.set_option("display.width", 250)
print(df.to_string(index=False))

## Итог

Запуск на T4, val, числа из `metrics.csv`, который пишет этот ноутбук (`MyDrive/avito_dataset/out/ft/e5-base/metrics.csv`). Прирост к BM25@1000 посчитан так же, как в ноутбуке 01.

| модель | объявление | запрос | lf@50 | lf@1000 | прирост к BM25@1000 |
|---|---|---|---|---|---|
| e5-base-ft | tpd | q | 0.8474 | 0.9487 | 0.056 |
| e5-base-ft | tpd | qf | 0.8557 | 0.9524 | 0.058 |

Дообучение подняло lf@50 с 0.8028 до 0.8557 и прирост к BM25 с 0.0509 до 0.058 (запрос с фильтрами, числа без дообучения из ноутбука 01). После дообучения фильтры в запросе стали помогать: 0.8557 против 0.8474 без них. В одиночку дообученная модель находит больше BM25: lf@1000 0.9524 против 0.9175 у BM25@1000.

## Скачать результаты

In [ ]:
# Как в ноутбуке 01: архив из сессии. Веса модели не берём, пайплайну нужны только эмбеддинги.
import zipfile
from google.colab import files
ZIP = f"/content/{MODEL_KEY}-ft.zip"
with zipfile.ZipFile(ZIP, "w", zipfile.ZIP_STORED) as z:
    for f in [f"items_{REP}.npy", "items_order.parquet", "q_val_qf.npy", "q_rank_qf.npy", "q_bench_qf.npy"]:
        z.write(f"{MOUT}/{f}", f"{MODEL_KEY}-ft/{f}")
files.download(ZIP)